# THUẬT TOÁN CÂY QUYẾT ĐỊNH ID3 (DECISION TREE)
**Bài toán:** Dự đoán khả năng chơi bóng (`play`) dựa trên điều kiện thời tiết

In [44]:
import numpy as np
import pandas as pd
from itertools import combinations

In [45]:

data = {
    'id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
    'outlook': ['sunny', 'sunny', 'overcast', 'rainy', 'rainy', 'rainy', 'overcast', 
                'sunny', 'sunny', 'rainy', 'sunny', 'overcast', 'overcast', 'rainy'],
    'temperature': ['hot', 'hot', 'hot', 'mild', 'cool', 'cool', 'cool', 
                    'mild', 'cool', 'mild', 'mild', 'mild', 'hot', 'mild'],
    'humidity': ['high', 'high', 'high', 'high', 'normal', 'normal', 'normal', 
                 'high', 'normal', 'normal', 'normal', 'high', 'normal', 'high'],
    'wind': ['weak', 'strong', 'weak', 'weak', 'weak', 'strong', 'strong', 
             'weak', 'weak', 'weak', 'strong', 'strong', 'weak', 'strong'],
    'play': ['no', 'no', 'yes', 'yes', 'yes', 'no', 'yes', 
             'no', 'yes', 'yes', 'yes', 'yes', 'yes', 'no']
}

df = pd.DataFrame(data).set_index('id')

df

,outlook,temperature,humidity,wind,play
id,,,,,
1,sunny,hot,high,weak,no
2,sunny,hot,high,strong,no
3,overcast,hot,high,weak,yes
4,rainy,mild,high,weak,yes
5,rainy,cool,normal,weak,yes
6,rainy,cool,normal,strong,no
7,overcast,cool,normal,strong,yes
8,sunny,mild,high,weak,no
9,sunny,cool,normal,weak,yes



- Hàm tính **Entropy**: $H(S) = -\sum p_i \ln(p_i)$
- Hàm tính **Information Gain**: $G(x, S) = H(S) - H(x, S)$

In [46]:
def entropy(target_col):
    values, counts = np.unique(target_col, return_counts=True)
    probabilities = counts / len(target_col)
    return -np.sum(probabilities * np.log(probabilities))

def information_gain(data, attribute, target_name="play"):
    total_entropy = entropy(data[target_name])
    values, counts = np.unique(data[attribute], return_counts=True)
    weighted_entropy = sum(
        (counts[i] / np.sum(counts)) * 
        entropy(data[data[attribute] == values[i]][target_name])
        for i in range(len(values))
    )
    
    gain = total_entropy - weighted_entropy
    return gain, weighted_entropy

In [47]:
features = ['outlook', 'temperature', 'humidity', 'wind']
target = 'play'

print(f"Entropy tập ban đầu H(S) = {entropy(df[target]):.4f}\n")

gain_table = []
for col in features:
    gain, h_val = information_gain(df, col, target)
    gain_table.append({'Thuộc tính': col, 'H(x, S)': round(h_val, 4), 'Information Gain': round(gain, 4)})

pd.DataFrame(gain_table).sort_values(by='Information Gain', ascending=False)

Entropy tập ban đầu H(S) = 0.6518



,Thuộc tính,"H(x, S)",Information Gain
0,outlook,0.4807,0.1710
2,humidity,0.5465,0.1052
3,wind,0.6184,0.0334
1,temperature,0.6315,0.0203


In [48]:
def id3(df_current, original_df, features, target_name="play", parent_node_class=None):
    if len(df_current) == 0:
        vals, counts = np.unique(original_df[target_name], return_counts=True)
        return vals[np.argmax(counts)]
    
    elif len(np.unique(df_current[target_name])) == 1:
        return np.unique(df_current[target_name])[0]

    elif len(features) == 0:
        return parent_node_class
    
    else:
       
        vals, counts = np.unique(df_current[target_name], return_counts=True)
        parent_node_class = vals[np.argmax(counts)]
        
       
        gains = [information_gain(df_current, f, target_name)[0] for f in features]
        best_feature = features[np.argmax(gains)]
        
       
        tree = {best_feature: {}}
        remaining_features = [f for f in features if f != best_feature]
        
        for val in np.unique(df_current[best_feature]):
            sub_data = df_current[df_current[best_feature] == val]
            subtree = id3(sub_data, original_df, remaining_features, target_name, parent_node_class)
            tree[best_feature][val] = subtree
            
        return tree


features = [col for col in df.columns if col != 'play']
tree_model = id3(df, df, features)

In [49]:
def display_tree(tree, indent=""):
    if not isinstance(tree, dict):
        print(f" ──> [Play = {tree.upper()}]")
        return
    for feature, branches in tree.items():
        for val, sub_tree in branches.items():
            print(f"{indent}├── ({feature} == '{val}')", end="")
            if isinstance(sub_tree, dict):
                print()
                display_tree(sub_tree, indent + "│   ")
            else:
                print(f" ──> [Play = {sub_tree.upper()}]")

print("CẤU TRÚC CÂY QUYẾT ĐỊNH (ID3):")
display_tree(tree_model)

CẤU TRÚC CÂY QUYẾT ĐỊNH (ID3):
├── (outlook == 'overcast') ──> [Play = YES]
├── (outlook == 'rainy')
│   ├── (wind == 'strong') ──> [Play = NO]
│   ├── (wind == 'weak') ──> [Play = YES]
├── (outlook == 'sunny')
│   ├── (humidity == 'high') ──> [Play = NO]
│   ├── (humidity == 'normal') ──> [Play = YES]


In [50]:
def predict(tree, sample):
    for feature in tree.keys():
        val = sample.get(feature)
        sub_tree = tree[feature].get(val)
        if isinstance(sub_tree, dict):
            return predict(sub_tree, sample)
        return sub_tree

# Ví dụ thử nghiệm 1 mẫu:
sample_test = {
    'outlook': 'sunny', 
    'temperature': 'cool', 
    'humidity': 'high', 
    'wind': 'weak'
}

result = predict(tree_model, sample_test)
print(f"Dữ liệu thời tiết: {sample_test}")
print(f"Kết quả dự đoán có chơi bóng không: {result.upper()}")

Dữ liệu thời tiết: {'outlook': 'sunny', 'temperature': 'cool', 'humidity': 'high', 'wind': 'weak'}
Kết quả dự đoán có chơi bóng không: NO


# BÀI TOÁN DỰ ĐOÁN RỦI RO TÍN DỤNG BẰNG THUẬT TOÁN ID3
**Nhãn mục tiêu (`Rủi ro tín dụng`):** 0 (Không rủi ro) hoặc 1 (Có rủi ro)

In [51]:
data = {
    'ID': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    'Độ tuổi': [25, 40, 35, 27, 31, 36, 48, 26, 33, 29, 38, 44, 42, 28, 30],
    'Hôn nhân': ['Độc thân', 'Đã kết hôn', 'Từng ly hôn', 'Đã kết hôn', 'Độc thân', 
                 'Đã kết hôn', 'Độc thân', 'Đã kết hôn', 'Từng ly hôn', 'Độc thân', 
                 'Đã kết hôn', 'Độc thân', 'Đã kết hôn', 'Độc thân', 'Đã kết hôn'],
    'Sở hữu BĐS': ['Ở cùng bố mẹ', 'Nhà sở hữu', 'Nhà thuê', 'Ở cùng bố mẹ', 'Nhà thuê', 
                  'Nhà sở hữu', 'Nhà thuê', 'Nhà thuê', 'Ở cùng bố mẹ', 'Nhà thuê', 
                  'Nhà sở hữu', 'Nhà sở hữu', 'Nhà sở hữu', 'Nhà thuê', 'Ở cùng bố mẹ'],
    'Thu nhập': [7000000, 18000000, 12000000, 9000000, 6000000, 
                 8000000, 7000000, 8000000, 5000000, 10000000, 
                 15000000, 14000000, 10000000, 7000000, 6000000],
    'Rủi ro tín dụng': [0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1]
}

df = pd.DataFrame(data).set_index('ID')
df


,Độ tuổi,Hôn nhân,Sở hữu BĐS,Thu nhập,Rủi ro tín dụng
ID,,,,,
1,25,Độc thân,Ở cùng bố mẹ,7000000,0
2,40,Đã kết hôn,Nhà sở hữu,18000000,0
3,35,Từng ly hôn,Nhà thuê,12000000,1
4,27,Đã kết hôn,Ở cùng bố mẹ,9000000,1
5,31,Độc thân,Nhà thuê,6000000,1
6,36,Đã kết hôn,Nhà sở hữu,8000000,1
7,48,Độc thân,Nhà thuê,7000000,0
8,26,Đã kết hôn,Nhà thuê,8000000,1
9,33,Từng ly hôn,Ở cùng bố mẹ,5000000,1


In [ ]:
def entropy(target_col):
    """H(S): Tính entropy của một cột mục tiêu"""
    values, counts = np.unique(target_col, return_counts=True)
    probabilities = counts / len(target_col)
    return -np.sum(probabilities * np.log(probabilities))

def get_best_split_continuous(data, feature, target_name="Rủi ro tín dụng"):
    """Tìm ngưỡng tốt nhất để phân tách một thuộc tính liên tục"""
    total_entropy = entropy(data[target_name])
    unique_vals = np.sort(data[feature].unique())
    if len(unique_vals) <= 1:
        return 0, None, total_entropy

    thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2.0
    best_gain = -1
    best_thresh = None
    best_h = total_entropy

    for t in thresholds:
        left = data[data[feature] <= t]
        right = data[data[feature] > t]
        
        h_split = (len(left) / len(data)) * entropy(left[target_name]) + \
                  (len(right) / len(data)) * entropy(right[target_name])
        gain = total_entropy - h_split
        if gain > best_gain:
            best_gain = gain
            best_thresh = t
            best_h = h_split

    return best_gain, best_thresh, best_h

def evaluate_features(data, features, target_name="Rủi ro tín dụng"):
    """Đánh giá toàn bộ các thuộc tính tại một nút"""
    total_entropy = entropy(data[target_name])
    results = {}

    for f in features:
        if np.issubdtype(data[f].dtype, np.number): 
            gain, thresh, h_val = get_best_split_continuous(data, f, target_name)
            results[f] = {'gain': gain, 'threshold': thresh, 'h': h_val, 'type': 'continuous'}
        else: 
            values, counts = np.unique(data[f], return_counts=True)
            weighted_h = sum(
                (counts[i] / len(data)) * entropy(data[data[f] == values[i]][target_name])
                for i in range(len(values))
            )
            gain = total_entropy - weighted_h
            results[f] = {'gain': gain, 'threshold': None, 'h': weighted_h, 'type': 'categorical'}

    return results

In [53]:
features = ['Độ tuổi', 'Hôn nhân', 'Sở hữu BĐS', 'Thu nhập']
target = 'Rủi ro tín dụng'

print(f"Entropy tập ban đầu H(S) = {entropy(df[target]):.4f}\n")

eval_res = evaluate_features(df, features, target)
summary_list = []
for k, v in eval_res.items():
    cond = f"<= {v['threshold']}" if v['type'] == 'continuous' else "Từng nhóm"
    summary_list.append({
        'Thuộc tính': k,
        'Điều kiện/Ngưỡng': cond,
        'H(x, S)': round(v['h'], 4),
        'Information Gain': round(v['gain'], 4)
    })

pd.DataFrame(summary_list).sort_values(by='Information Gain', ascending=False)

Entropy tập ban đầu H(S) = 0.6730



,Thuộc tính,Điều kiện/Ngưỡng,"H(x, S)",Information Gain
0,Độ tuổi,<= 37.0,0.5004,0.1726
3,Thu nhập,<= 14500000.0,0.5349,0.1381
1,Hôn nhân,Từng nhóm,0.5959,0.0771
2,Sở hữu BĐS,Từng nhóm,0.6289,0.0441


In [ ]:
def id3_extended(df_curr, orig_df, features, target_name="Rủi ro tín dụng", parent_class=None):
 
    if len(df_curr) == 0:
        vals, counts = np.unique(orig_df[target_name], return_counts=True)
        return vals[np.argmax(counts)]

    elif len(np.unique(df_curr[target_name])) <= 1:
        return np.unique(df_curr[target_name])[0]

    elif len(features) == 0:
        return parent_class

    else:
        vals, counts = np.unique(df_curr[target_name], return_counts=True)
        parent_class = vals[np.argmax(counts)]

        # Đánh giá các thuộc tính
        eval_dict = evaluate_features(df_curr, features, target_name)
        
        # Chọn thuộc tính có Gain lớn nhất
        best_f = max(eval_dict, key=lambda x: eval_dict[x]['gain'])
        best_info = eval_dict[best_f]

        # Nếu Gain bằng 0 (không chia tách thêm được thông tin)
        if best_info['gain'] <= 1e-6:
            return parent_class

        tree = {}
        if best_info['type'] == 'categorical':
            node_key = best_f
            tree[node_key] = {}
            remaining_f = [f for f in features if f != best_f]
            for val in df_curr[best_f].unique():
                sub_df = df_curr[df_curr[best_f] == val]
                tree[node_key][val] = id3_extended(sub_df, orig_df, remaining_f, target_name, parent_class)
        else:
            thresh = best_info['threshold']
            node_key = f"{best_f} <= {thresh:g}"
            tree[node_key] = {}
            # Nhánh <= thresh
            left_df = df_curr[df_curr[best_f] <= thresh]
            tree[node_key]["Yes"] = id3_extended(left_df, orig_df, features, target_name, parent_class)
            # Nhánh > thresh
            right_df = df_curr[df_curr[best_f] > thresh]
            tree[node_key]["No"] = id3_extended(right_df, orig_df, features, target_name, parent_class)

        return tree

# Huấn luyện mô hình cây
tree_credit = id3_extended(df, df, features)

In [55]:
def print_tree(tree, indent=""):
    if not isinstance(tree, dict):
        print(f" ──> [Rủi ro = {tree}]")
        return
    for node, branches in tree.items():
        for val, subtree in branches.items():
            print(f"{indent}├── ({node} == '{val}')", end="")
            if isinstance(subtree, dict):
                print()
                print_tree(subtree, indent + "│   ")
            else:
                print(f" ──> [Rủi ro = {subtree}]")

print("CẤU TRÚC CÂY QUYẾT ĐỊNH ID3:")
print_tree(tree_credit)

CẤU TRÚC CÂY QUYẾT ĐỊNH ID3:
├── (Độ tuổi <= 37 == 'Yes')
│   ├── (Hôn nhân == 'Độc thân')
│   │   ├── (Độ tuổi <= 26.5 == 'Yes') ──> [Rủi ro = 0]
│   │   ├── (Độ tuổi <= 26.5 == 'No')
│   │   │   ├── (Thu nhập <= 8.5e+06 == 'Yes') ──> [Rủi ro = 1]
│   │   │   ├── (Thu nhập <= 8.5e+06 == 'No') ──> [Rủi ro = 0]
│   ├── (Hôn nhân == 'Từng ly hôn') ──> [Rủi ro = 1]
│   ├── (Hôn nhân == 'Đã kết hôn') ──> [Rủi ro = 1]
├── (Độ tuổi <= 37 == 'No')
│   ├── (Độ tuổi <= 43 == 'Yes') ──> [Rủi ro = 0]
│   ├── (Độ tuổi <= 43 == 'No')
│   │   ├── (Độ tuổi <= 46 == 'Yes') ──> [Rủi ro = 1]
│   │   ├── (Độ tuổi <= 46 == 'No') ──> [Rủi ro = 0]


In [56]:
def predict_credit(tree, sample):
    if not isinstance(tree, dict):
        return tree
    
    node = list(tree.keys())[0]
    
    # Kiểm tra nút điều kiện liên tục (ví dụ: 'Thu nhập <= 1.1e+07')
    if "<=" in node:
        feat, thresh = node.split(" <= ")
        thresh = float(thresh)
        val = sample[feat]
        branch = "Yes" if val <= thresh else "No"
        return predict_credit(tree[node][branch], sample)
    else: # Nút biến phân loại
        val = sample[node]
        if val in tree[node]:
            return predict_credit(tree[node][val], sample)
        return "Không có nhánh khớp"

# Test mẫu khách hàng mới
sample_customer = {
    'Độ tuổi': 32,
    'Hôn nhân': 'Đã kết hôn',
    'Sở hữu BĐS': 'Nhà thuê',
    'Thu nhập': 12000000
}

pred = predict_credit(tree_credit, sample_customer)
print(f"Thông tin khách hàng: {sample_customer}")
print(f"Dự đoán rủi ro tín dụng (0: Không, 1: Có): {pred}")

Thông tin khách hàng: {'Độ tuổi': 32, 'Hôn nhân': 'Đã kết hôn', 'Sở hữu BĐS': 'Nhà thuê', 'Thu nhập': 12000000}
Dự đoán rủi ro tín dụng (0: Không, 1: Có): 1


# CÂY QUYẾT ĐỊNH CART (CLASSIFICATION AND REGRESSION TREES) VỚI GINI INDEX
**Bài toán:** Phân loại Rủi ro tín dụng $\in \{0, 1\}$[cite: 3]
- **Chỉ số Gini của tập $S$:**
  $$Gini(S) = 1 - \sum_{i=1}^C p_i^2$$
- **Chỉ số Gini khi phân chia theo thuộc tính/ngưỡng $A$:**
  $$Gini_A(S) = \frac{|S_1|}{|S|} Gini(S_1) + \frac{|S_2|}{|S|} Gini(S_2)$$
- CART luôn phân nhánh **nhị phân (binary split)** và chọn phép phân chia có **$Gini_A(S)$ nhỏ nhất** (tương đương Gini Gain lớn nhất).

In [57]:
def gini_impurity(target_col):
    """Tính chỉ số Gini: Gini(S) = 1 - sum(p_i^2)"""
    if len(target_col) == 0:
        return 0
    _, counts = np.unique(target_col, return_counts=True)
    probs = counts / len(target_col)
    return 1.0 - np.sum(probs ** 2)

In [ ]:
def evaluate_best_cart_split(data, feature, target_name="Rủi ro tín dụng"):
    """
    Tìm phép chia nhị phân tối ưu cho thuộc tính feature:
    - Thuộc tính liên tục: chia theo dạng (<= threshold)
    - Thuộc tính phân loại: chia theo tập con nhị phân (in subset)
    """
    total_len = len(data)
    best_gini = 1.0
    best_condition = None
    best_subsets = None

    if np.issubdtype(data[feature].dtype, np.number):
        unique_vals = np.sort(data[feature].unique())
        if len(unique_vals) <= 1:
            return 1.0, None, None
            
        thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2.0
        for t in thresholds:
            left = data[data[feature] <= t]
            right = data[data[feature] > t]
            
            gini_split = (len(left)/total_len) * gini_impurity(left[target_name]) + \
                         (len(right)/total_len) * gini_impurity(right[target_name])
            
            if gini_split < best_gini:
                best_gini = gini_split
                best_condition = ('num', t)
                best_subsets = (left, right)
    else: 
        unique_vals = list(data[feature].unique())
        if len(unique_vals) <= 1:
            return 1.0, None, None
            
    
        for r in range(1, len(unique_vals) // 2 + 1):
            for subset in combinations(unique_vals, r):
                subset = list(subset)
                left = data[data[feature].isin(subset)]
                right = data[~data[feature].isin(subset)]
                
                gini_split = (len(left)/total_len) * gini_impurity(left[target_name]) + \
                             (len(right)/total_len) * gini_impurity(right[target_name])
                
                if gini_split < best_gini:
                    best_gini = gini_split
                    best_condition = ('cat', subset)
                    best_subsets = (left, right)

    return best_gini, best_condition, best_subsets

In [59]:
features = ['Độ tuổi', 'Hôn nhân', 'Sở hữu BĐS', 'Thu nhập']
target = 'Rủi ro tín dụng'

print(f"Gini nút gốc: {gini_impurity(df[target]):.4f}\n")

root_eval = []
for f in features:
    gini_val, cond, _ = evaluate_best_cart_split(df, f, target)
    if cond:
        cond_str = f"<= {cond[1]:g}" if cond[0] == 'num' else f"thuộc {cond[1]}"
    else:
        cond_str = "None"
    root_eval.append({
        'Thuộc tính': f,
        'Điều kiện rẽ nhánh (Nhánh Trái)': cond_str,
        'Gini Split': round(gini_val, 4)
    })

pd.DataFrame(root_eval).sort_values(by='Gini Split', ascending=True)

Gini nút gốc: 0.4800



,Thuộc tính,Điều kiện rẽ nhánh (Nhánh Trái),Gini Split
0,Độ tuổi,<= 37,0.3200
3,Thu nhập,<= 1.45e+07,0.3692
1,Hôn nhân,thuộc ['Từng ly hôn'],0.4308
2,Sở hữu BĐS,thuộc ['Nhà sở hữu'],0.4400


In [60]:
def cart(data, orig_data, features, target_name="Rủi ro tín dụng", max_depth=5, current_depth=0):
    # 1. Điều kiện dừng: Tập rỗng
    if len(data) == 0:
        vals, counts = np.unique(orig_data[target_name], return_counts=True)
        return vals[np.argmax(counts)]

    # 2. Điều kiện dừng: Thuần khiết (Gini = 0) hoặc đạt độ sâu tối đa
    if gini_impurity(data[target_name]) == 0 or current_depth >= max_depth:
        vals, counts = np.unique(data[target_name], return_counts=True)
        return vals[np.argmax(counts)]

    # Tìm thuộc tính và phép phân chia tốt nhất (Gini split nhỏ nhất)
    best_gini = 1.0
    best_feature = None
    best_cond = None
    best_splits = None

    for f in features:
        gini_split, cond, splits = evaluate_best_cart_split(data, f, target_name)
        if splits is not None and gini_split < best_gini:
            best_gini = gini_split
            best_feature = f
            best_cond = cond
            best_splits = splits

    # Nếu không cải thiện được Gini (không tách thêm được mẫu)
    current_gini = gini_impurity(data[target_name])
    if best_feature is None or best_gini >= current_gini:
        vals, counts = np.unique(data[target_name], return_counts=True)
        return vals[np.argmax(counts)]

    # Tạo nút cây
    left_df, right_df = best_splits
    tree = {}

    if best_cond[0] == 'num':
        rule_left = f"{best_feature} <= {best_cond[1]:g}"
        rule_right = f"{best_feature} > {best_cond[1]:g}"
    else:
        rule_left = f"{best_feature} in {best_cond[1]}"
        rule_right = f"{best_feature} not in {best_cond[1]}"

    node_label = f"Check: {best_feature}"
    tree[node_label] = {
        rule_left: cart(left_df, orig_data, features, target_name, max_depth, current_depth + 1),
        rule_right: cart(right_df, orig_data, features, target_name, max_depth, current_depth + 1)
    }

    return tree

# Huấn luyện mô hình CART
cart_tree = cart(df, df, features)

In [61]:
def print_cart_tree(tree, indent=""):
    if not isinstance(tree, dict):
        print(f" ──> [Rủi ro tín dụng = {tree}]")
        return
    for node, branches in tree.items():
        for cond, subtree in branches.items():
            print(f"{indent}├── IF ({cond})", end="")
            if isinstance(subtree, dict):
                print()
                print_cart_tree(subtree, indent + "│   ")
            else:
                print(f" ──> [Rủi ro tín dụng = {subtree}]")

print("CẤU TRÚC CÂY QUYẾT ĐỊNH CART (GINI INDEX):")
print_cart_tree(cart_tree)

CẤU TRÚC CÂY QUYẾT ĐỊNH CART (GINI INDEX):
├── IF (Độ tuổi <= 37)
│   ├── IF (Độ tuổi <= 25.5) ──> [Rủi ro tín dụng = 0]
│   ├── IF (Độ tuổi > 25.5)
│   │   ├── IF (Thu nhập <= 9.5e+06) ──> [Rủi ro tín dụng = 1]
│   │   ├── IF (Thu nhập > 9.5e+06)
│   │   │   ├── IF (Độ tuổi <= 32) ──> [Rủi ro tín dụng = 0]
│   │   │   ├── IF (Độ tuổi > 32) ──> [Rủi ro tín dụng = 1]
├── IF (Độ tuổi > 37)
│   ├── IF (Độ tuổi <= 43) ──> [Rủi ro tín dụng = 0]
│   ├── IF (Độ tuổi > 43)
│   │   ├── IF (Độ tuổi <= 46) ──> [Rủi ro tín dụng = 1]
│   │   ├── IF (Độ tuổi > 46) ──> [Rủi ro tín dụng = 0]


In [62]:
def predict_cart(tree, sample):
    if not isinstance(tree, dict):
        return tree
    
    node = list(tree.keys())[0]
    branches = tree[node]
    
    for rule, subtree in branches.items():
        if "<=" in rule:
            feat, val_str = rule.split(" <= ")
            if sample[feat] <= float(val_str):
                return predict_cart(subtree, sample)
        elif ">" in rule:
            feat, val_str = rule.split(" > ")
            if sample[feat] > float(val_str):
                return predict_cart(subtree, sample)
        elif " not in " in rule:
            feat, list_str = rule.split(" not in ")
            subset = eval(list_str)
            if sample[feat] not in subset:
                return predict_cart(subtree, sample)
        elif " in " in rule:
            feat, list_str = rule.split(" in ")
            subset = eval(list_str)
            if sample[feat] in subset:
                return predict_cart(subtree, sample)
                
    return "Không xác định"

# Thử nghiệm mẫu khách hàng
sample_customer = {
    'Độ tuổi': 35,
    'Hôn nhân': 'Đã kết hôn',
    'Sở hữu BĐS': 'Nhà sở hữu',
    'Thu nhập': 12000000
}

pred = predict_cart(cart_tree, sample_customer)
print(f"Khách hàng: {sample_customer}")
print(f"Dự đoán Rủi ro tín dụng: {pred}")

Khách hàng: {'Độ tuổi': 35, 'Hôn nhân': 'Đã kết hôn', 'Sở hữu BĐS': 'Nhà sở hữu', 'Thu nhập': 12000000}
Dự đoán Rủi ro tín dụng: 1
